# Fine-tuning MedSAM3 on the 2018 Data Science Bowl (BBBC038) nuclei dataset

This notebook fine-tunes **SAM3** (via LoRA, the MedSAM3 recipe) for **cell-nucleus
segmentation** on [`MedOtter/2018-Data-Science-Bowl`](https://huggingface.co/datasets/MedOtter/2018-Data-Science-Bowl),
and can run on **GPU or TPU** (TPU via [AutoXLA](https://github.com/Locutusque/autoxla)).

### What this notebook does
1. Loads the Hugging Face dataset (RGB microscopy images + binary nucleus masks).
2. **Derives instance annotations from the binary semantic masks.** The dataset ships a
   *binary semantic* mask per image (the union of all nuclei), but SAM3 is a DETR-style
   detector that trains on *per-object* boxes + masks + a text query. We recover
   per-nucleus instances with **connected-components labelling** (`scipy.ndimage.label`),
   which is the standard weak-instance approximation for DSB2018.
3. Wraps each image as a SAM3 `Datapoint` with a single `"nucleus"` text query whose
   targets are all the derived instances.
4. Builds SAM3, applies LoRA, and trains with the native SAM3 loss (Hungarian matching,
   box + mask + classification losses).
5. Optionally runs the whole thing on **TPU** through AutoXLA (SPMD sharding, optional
   QLoRA-style base-model quantization).
6. Saves LoRA weights and runs a quick qualitative inference on the test split.

> **Instance-mask caveat.** Connected components merge nuclei that physically touch, so a
> cluster of touching nuclei becomes one instance. This is fine for *semantic* nucleus
> segmentation and is a reasonable weak signal for detection. For true instance masks, use
> the original per-nucleus PNGs from [BBBC038](https://bbbc.broadinstitute.org/BBBC038).

## 1. Environment setup

The cell below is environment-aware (Kaggle / Colab / local). On Kaggle or Colab it clones
the MedSAM3 repo — which provides the `sam3` package, `lora_layers.py`, and `tpu_utils.py` —
and installs dependencies. If you already run inside a MedSAM3 checkout, it uses that.

### Running on Kaggle (upload this notebook and Run All)

In the right-hand **Session options** panel:

1. **Accelerator → GPU** (T4 ×2 or P100).
2. **Internet → On** (required to clone the repo and download the model + dataset).
3. **Add-ons → Secrets** → add a secret named **`HF_TOKEN`** holding a Hugging Face token,
   and accept the SAM3 model licence at <https://huggingface.co/facebook/sam3>
   (SAM3 weights are gated).

Then **Run All** — no code edits required.

In [ ]:
import os, sys, subprocess

IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

# MedSAM3 repo that carries this notebook, tpu_utils.py, and the sam3 package.
# Switch MEDSAM3_BRANCH to "main" once the feature branch is merged.
MEDSAM3_REPO = "https://github.com/Locutusque/MedSAM3.git"
MEDSAM3_BRANCH = "claude/kaggle-nuclei-notebook"

def _is_repo_root(p):
    return (os.path.exists(os.path.join(p, "sam3"))
            and os.path.exists(os.path.join(p, "tpu_utils.py")))

def _has_internet(url="https://huggingface.co", timeout=6):
    import urllib.request
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False

if _is_repo_root(os.getcwd()):
    MEDSAM3_DIR = os.getcwd()                        # already inside the repo
else:
    base = "/kaggle/working" if IS_KAGGLE else os.getcwd()
    MEDSAM3_DIR = os.path.join(base, "MedSAM3")
    if not _is_repo_root(MEDSAM3_DIR):
        if not _has_internet():
            raise RuntimeError(
                "No internet access. On Kaggle enable it via "
                "Session options -> Internet -> On, then re-run this cell.")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", MEDSAM3_BRANCH,
                        MEDSAM3_REPO, MEDSAM3_DIR], check=True)

sys.path.insert(0, MEDSAM3_DIR)
os.chdir(MEDSAM3_DIR)
print(f"Environment: {ENV} | repo: {MEDSAM3_DIR}")

### Dependencies

Installs SAM3's requirements (iopath, hydra-core, omegaconf, decord, einops,
open-clip-torch, ...). On Kaggle/Colab, `torch`/`torchvision` are **not** reinstalled — the
preinstalled CUDA build is kept — while everything else is installed from `requirements.txt`.

In [ ]:
import re

reqs = []
if os.path.exists("requirements.txt"):
    for line in open("requirements.txt"):
        line = line.split("#", 1)[0].strip()
        if not line:
            continue
        # keep the platform's working CUDA torch on managed runtimes
        if (IS_KAGGLE or IS_COLAB) and re.match(r"(?i)^(torch|torchvision)\b", line):
            continue
        reqs.append(line)
# notebook-only helpers (usually already present on Kaggle)
reqs += ["datasets", "scipy", "matplotlib", "pycocotools"]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=False)
print(f"Installed {len(reqs)} dependency specs")

### Hugging Face authentication

SAM3 weights are gated, so `build_sam3_image_model(load_from_HF=True)` needs a token.
On Kaggle this is read from **Secrets** (`HF_TOKEN`); locally, run `huggingface-cli login`
or set the `HF_TOKEN` environment variable.

In [ ]:
def hf_login():
    token = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
             or os.environ.get("HUGGINGFACE_HUB_TOKEN"))
    if token is None and IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            client = UserSecretsClient()
            for key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN", "hf_token"):
                try:
                    token = client.get_secret(key)
                    if token:
                        break
                except Exception:
                    continue
        except Exception:
            pass
    if token:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        print("Hugging Face login OK")
    else:
        print("No Hugging Face token found. SAM3 weights are gated:\n"
              "  * Kaggle: Add-ons -> Secrets -> add HF_TOKEN\n"
              "  * local : run `huggingface-cli login` or set HF_TOKEN\n"
              "  then accept the licence at https://huggingface.co/facebook/sam3")

hf_login()

### (Optional) TPU + AutoXLA install

**Leave `INSTALL_TPU = False` for the GPU path** (the recommended way to run on Kaggle —
select the **GPU** accelerator). `tpu_utils.py` degrades to a no-op without `torch_xla`, so
skipping this changes nothing and training runs on GPU.

Set `INSTALL_TPU = True` **only** on a TPU VM (a Kaggle **TPU VM** accelerator session, a
Google Cloud TPU VM, ...). It installs `torch_xla` + AutoXLA. AutoXLA is installed
**non-editable** (`pip install ./autoxla`, not `-e`) so the freshly built package is
importable in this already-running kernel — an editable install adds a `.pth` finder that a
live kernel won't load until restart.

In [ ]:
# Only run on a TPU VM.
INSTALL_TPU = False  # set True on a TPU VM

# AutoXLA's TPU segmentation support + the packaging fix that makes it
# pip-installable live on this branch until they merge to main. Switch to
# "main" once https://github.com/Locutusque/autoxla/pull/1 is merged.
AUTOXLA_REPO = "https://github.com/Locutusque/autoxla.git"
AUTOXLA_BRANCH = "claude/image-segmentation-quantization-l5h9x9"

if INSTALL_TPU:
    import os, sys, subprocess, importlib
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch~=2.8.0"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_xla[tpu]~=2.8.0",
                    "--find-links=https://storage.googleapis.com/libtpu-releases/index.html",
                    "--find-links=https://storage.googleapis.com/libtpu-wheels/index.html"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_xla[pallas]",
                    "--find-links=https://storage.googleapis.com/jax-releases/jax_nightly_releases.html",
                    "--find-links=https://storage.googleapis.com/jax-releases/jaxlib_nightly_releases.html"], check=False)
    # AutoXLA (segmentation + quantization support used by tpu_utils.prepare_model_for_tpu).
    # Installed from source; the flat-root package now installs correctly.
    if not os.path.exists("autoxla"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", AUTOXLA_BRANCH,
                        AUTOXLA_REPO, "autoxla"], check=True)
    # Use a NON-editable install: it lands in site-packages (already on sys.path)
    # and so is importable in THIS running kernel after invalidate_caches(). An
    # editable `-e` install registers a .pth/finder that a live kernel does not
    # pick up, so `import AutoXLA` fails until a restart. --no-deps because the
    # TPU steps above already provide torch/torch_xla/jax and transformers came
    # from requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "./autoxla"], check=True)
    importlib.invalidate_caches()
    try:
        import site
        for sp in (site.getsitepackages() if hasattr(site, "getsitepackages") else []):
            if sp not in sys.path:
                sys.path.insert(0, sp)
    except Exception:
        pass
    importlib.invalidate_caches()
    if importlib.util.find_spec("AutoXLA") is None:
        raise RuntimeError("AutoXLA is not importable after install — check the pip output above.")
    print("AutoXLA installed:", importlib.util.find_spec("AutoXLA").origin)

## 2. Configuration

Small defaults so the notebook runs end-to-end quickly. Scale `NUM_EPOCHS`, `BATCH_SIZE`
and set `MAX_TRAIN_SAMPLES = None` for a real run.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class Cfg:
    # data
    hf_dataset: str = "MedOtter/2018-Data-Science-Bowl"
    train_split: str = "train"          # stage1_train (native masks)
    val_split: str = "stage1_test"      # has GT masks (RLE-decoded in the HF repo)
    query_text: str = "nucleus"
    resolution: int = 1008              # SAM3 input resolution (matches MedSAM3)
    min_instance_area: int = 8          # drop connected components smaller than this (px, orig res)
    max_instances: int = 256            # cap objects/image to bound memory (keeps the largest)
    max_train_samples: Optional[int] = 64   # None = full 670-image train split
    max_val_samples: Optional[int] = 16     # None = full val split

    # training
    num_epochs: int = 5
    batch_size: int = 1
    num_workers: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    mixed_precision: str = "bf16"       # "bf16" enables autocast on TPU/GPU
    seed: int = 42
    grad_clip: float = 1.0

    # LoRA (MedSAM3 defaults)
    lora_rank: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "out_proj",
        "qkv", "proj", "fc1", "fc2", "c_fc", "c_proj", "linear1", "linear2",
    ])
    apply_to_vision_encoder: bool = True
    apply_to_text_encoder: bool = True
    apply_to_geometry_encoder: bool = True
    apply_to_detr_encoder: bool = True
    apply_to_detr_decoder: bool = True
    apply_to_mask_decoder: bool = True

    # device / TPU
    device: str = "auto"                # "auto" | "cuda" | "cpu" | "tpu"
    tpu_sharding_strategy: str = "fsdp"
    tpu_use_fsdp_wrap: bool = False     # keep False for LoRA training
    quantize_base_model: bool = False   # QLoRA-style int8/int4 of the frozen base (TPU)
    quant_n_bits: int = 8
    quant_use_pallas: bool = True

    # output
    output_dir: str = "outputs/medsam3_dsb2018_nuclei"

cfg = Cfg()

import torch, numpy as np, random
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
print(cfg)

### Resolve the device (GPU / TPU / CPU)

`tpu_utils` is imported unconditionally; it is a no-op without `torch_xla`, so the exact
same training loop runs on GPU and TPU.

In [ ]:
import tpu_utils

USE_TPU = (cfg.device == "tpu") or (cfg.device == "auto" and tpu_utils.is_tpu_available())

if USE_TPU:
    tpu_utils.require_tpu()
    device = tpu_utils.get_xla_device()
    build_device = "cpu"     # build on host; AutoXLA moves + shards the model
    print(f"Using TPU via AutoXLA ({tpu_utils.world_size()} cores, XLA SPMD)")
elif cfg.device in ("cuda", "auto") and torch.cuda.is_available():
    device = torch.device("cuda"); build_device = "cuda"
    print("Using CUDA:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu"); build_device = "cpu"
    print("Using CPU (slow; for a smoke test only)")

use_autocast = cfg.mixed_precision.lower() in ("bf16", "bfloat16")

## 3. Load the Hugging Face dataset

Columns (per the dataset card): `image` (RGB), `mask` (binary `L`, values `{0, 255}` — the
union of all nuclei), plus `image_id`, `num_nuclei`, `height`, `width`, `split`, `usage`,
`is_grayscale`.

In [ ]:
from datasets import load_dataset

hf_train = load_dataset(cfg.hf_dataset, split=cfg.train_split)
try:
    hf_val = load_dataset(cfg.hf_dataset, split=cfg.val_split)
except Exception as e:
    print(f"Validation split '{cfg.val_split}' unavailable ({e}); training without validation.")
    hf_val = None

if cfg.max_train_samples:
    hf_train = hf_train.select(range(min(cfg.max_train_samples, len(hf_train))))
if hf_val is not None and cfg.max_val_samples:
    hf_val = hf_val.select(range(min(cfg.max_val_samples, len(hf_val))))

print("train images:", len(hf_train), "| val images:", (len(hf_val) if hf_val else 0))
print("columns:", hf_train.column_names)
ex0 = hf_train[0]
print("example:", {k: ex0[k] for k in ("image_id", "num_nuclei", "height", "width", "is_grayscale")})
print("image mode:", ex0["image"].mode, "| mask mode:", ex0["mask"].mode)

## 4. Binary semantic mask → per-nucleus instances

`mask_to_instances` labels connected components of the binary mask (at the original
resolution, where instances are cleanest), then for each component returns:
- a **box** in normalized `CxCyWH` format (the format SAM3 consumes, matching the repo's
  `COCOSegmentDataset`),
- the pixel **area**,
- a per-instance binary **mask** resized to the model resolution.

Tiny components (`< min_instance_area`) are dropped as noise, and only the largest
`max_instances` are kept to bound memory on very dense images.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage

def mask_to_instances(mask_np, orig_w, orig_h, resolution, min_area, max_instances):
    '''Return list of (bbox_cxcywh_norm[float32,4], area_px, segment[res,res] bool).'''
    binm = (np.asarray(mask_np) > 127).astype(np.uint8)
    if binm.ndim == 3:              # safety: collapse accidental channels
        binm = binm[..., 0]
    labels, n = ndimage.label(binm)
    if n == 0:
        return []
    slices = ndimage.find_objects(labels)
    out = []
    for i, sl in enumerate(slices, start=1):
        if sl is None:
            continue
        comp = labels[sl] == i
        area_px = int(comp.sum())
        if area_px < min_area:
            continue
        ys, xs = sl
        bw, bh = (xs.stop - xs.start), (ys.stop - ys.start)
        cx, cy = xs.start + bw / 2.0, ys.start + bh / 2.0
        bbox = torch.tensor([cx / orig_w, cy / orig_h, bw / orig_w, bh / orig_h],
                            dtype=torch.float32)
        full = np.zeros((orig_h, orig_w), dtype=np.uint8)
        full[sl][comp] = 1
        seg = F.interpolate(torch.from_numpy(full).float()[None, None],
                            size=(resolution, resolution), mode="nearest").squeeze() > 0.5
        out.append((bbox, float(area_px), seg, area_px))
    out.sort(key=lambda t: t[3], reverse=True)      # keep the largest components
    out = out[:max_instances]
    return [(b, a, s) for (b, a, s, _) in out]

# quick self-check on the first training mask
_m = np.array(hf_train[0]["mask"])
_inst = mask_to_instances(_m, _m.shape[1], _m.shape[0], cfg.resolution,
                          cfg.min_instance_area, cfg.max_instances)
print(f"derived {len(_inst)} instances (dataset reports num_nuclei={hf_train[0]['num_nuclei']})")

## 5. `NucleiHFDataset` — SAM3 `Datapoint`s from the HF dataset

This mirrors the output of the repo's `COCOSegmentDataset` exactly (same image transform,
same normalized `CxCyWH` boxes, same `Datapoint` structure) but sources images/masks from
the HF dataset and derives objects via `mask_to_instances`. Each image gets a **single**
`"nucleus"` `FindQueryLoaded` whose `object_ids_output` lists all instances (`is_exhaustive=True`
since the union mask annotates every nucleus).

In [ ]:
from collections import defaultdict
from PIL import Image as PILImage
from torch.utils.data import Dataset
from torchvision.transforms import v2

from sam3.train.data.sam3_image_dataset import (
    Datapoint, Image, Object, FindQueryLoaded, InferenceMetadata,
)

class NucleiHFDataset(Dataset):
    def __init__(self, hf_ds, cfg):
        self.ds = hf_ds
        self.cfg = cfg
        self.resolution = cfg.resolution
        self.transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        rec = self.ds[idx]
        pil = rec["image"].convert("RGB")
        orig_w, orig_h = pil.size
        mask_np = np.array(rec["mask"].convert("L"))

        # instances BEFORE resizing the image (cleanest labelling), boxes normalized
        instances = mask_to_instances(
            mask_np, orig_w, orig_h, self.resolution,
            self.cfg.min_instance_area, self.cfg.max_instances)

        pil_resized = pil.resize((self.resolution, self.resolution), PILImage.BILINEAR)
        image_tensor = self.transform(pil_resized)

        objects = []
        for obj_id, (bbox, area, seg) in enumerate(instances):
            objects.append(Object(bbox=bbox, area=area, object_id=obj_id, segment=seg))

        meta = InferenceMetadata(
            coco_image_id=idx, original_image_id=idx, original_category_id=0,
            original_size=(orig_h, orig_w), object_id=-1, frame_index=-1,
        )
        query = FindQueryLoaded(
            query_text=self.cfg.query_text,
            image_id=0,
            object_ids_output=[o.object_id for o in objects],
            is_exhaustive=True,
            query_processing_order=0,
            inference_metadata=meta,
        )
        image_obj = Image(data=image_tensor, objects=objects,
                          size=(self.resolution, self.resolution))
        return Datapoint(find_queries=[query], images=[image_obj], raw_images=[pil_resized])

train_ds = NucleiHFDataset(hf_train, cfg)
val_ds = NucleiHFDataset(hf_val, cfg) if hf_val is not None else None
print("train datapoints:", len(train_ds), "| val datapoints:", (len(val_ds) if val_ds else 0))

### Sanity-check a sample

Overlay the derived instance boxes on the image and show the semantic mask. If boxes don't
line up with nuclei, revisit `min_instance_area` / `max_instances`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

_rec = hf_train[0]
_pil = _rec["image"].convert("RGB")
_ow, _oh = _pil.size
_inst = mask_to_instances(np.array(_rec["mask"].convert("L")), _ow, _oh,
                          cfg.resolution, cfg.min_instance_area, cfg.max_instances)

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(_pil); ax[0].set_title(f"image ({_ow}x{_oh})"); ax[0].axis("off")
ax[1].imshow(_rec["mask"], cmap="gray"); ax[1].set_title("binary semantic mask"); ax[1].axis("off")
ax[2].imshow(_pil); ax[2].set_title(f"{len(_inst)} derived instance boxes"); ax[2].axis("off")
for bbox, area, seg in _inst:
    cx, cy, bw, bh = bbox.tolist()
    x, y, w, h = (cx - bw/2)*_ow, (cy - bh/2)*_oh, bw*_ow, bh*_oh
    ax[2].add_patch(patches.Rectangle((x, y), w, h, lw=1, ec="lime", fc="none"))
plt.tight_layout(); plt.show()

## 6. Build SAM3 + LoRA

Build the base model (on CPU when targeting TPU; AutoXLA moves + shards it afterwards),
then apply LoRA with the MedSAM3 configuration. LoRA freezes the base and adds trainable
low-rank adapters to the targeted projections.

In [ ]:
from sam3.model_builder import build_sam3_image_model
from lora_layers import LoRAConfig, apply_lora_to_model, save_lora_weights, count_parameters

print("Building SAM3 (downloads weights from HF on first run)...")
model = build_sam3_image_model(
    device=build_device, compile=False, load_from_HF=True,
    bpe_path="sam3/assets/bpe_simple_vocab_16e6.txt.gz", eval_mode=False,
)

lora_config = LoRAConfig(
    rank=cfg.lora_rank, alpha=cfg.lora_alpha, dropout=cfg.lora_dropout,
    target_modules=cfg.lora_target_modules,
    apply_to_vision_encoder=cfg.apply_to_vision_encoder,
    apply_to_text_encoder=cfg.apply_to_text_encoder,
    apply_to_geometry_encoder=cfg.apply_to_geometry_encoder,
    apply_to_detr_encoder=cfg.apply_to_detr_encoder,
    apply_to_detr_decoder=cfg.apply_to_detr_decoder,
    apply_to_mask_decoder=cfg.apply_to_mask_decoder,
)
model = apply_lora_to_model(model, lora_config)

stats = count_parameters(model)
print(f"Trainable params: {stats['trainable_parameters']:,} ({stats['trainable_percentage']:.2f}%)")

### Move to device / prepare for TPU

On TPU, `tpu_utils.prepare_model_for_tpu` runs AutoXLA's *quantize → XLA device →
SPMD-shard* pipeline (must run **after** LoRA and **before** the optimizer is created).
On GPU/CPU we just `.to(device)`.

In [ ]:
if USE_TPU:
    tpu_cfg = {
        "sharding_strategy": cfg.tpu_sharding_strategy,
        "use_fsdp_wrap": cfg.tpu_use_fsdp_wrap,
        "quantize_base_model": cfg.quantize_base_model,
        "quantization": {"n_bits": cfg.quant_n_bits, "use_pallas": cfg.quant_use_pallas},
    }
    print("Preparing model for TPU via AutoXLA...")
    model = tpu_utils.prepare_model_for_tpu(model, tpu_cfg)
else:
    model.to(device)

# unwrapped handle for SAM3 helper methods (works whether or not FSDPv2-wrapped)
unwrapped = getattr(model, "module", model)

## 7. Loss, matcher and optimizer

Uses SAM3's native training objective (identical weights/config to
`train_sam3_lora_native.py`): Hungarian matching + box (L1 + GIoU), classification
(focal / IA-BCE with presence), and mask (focal + dice) losses.

In [ ]:
from torch.optim import AdamW
from sam3.model.model_misc import SAM3Output
from sam3.train.loss.loss_fns import IABCEMdetr, Boxes, Masks, CORE_LOSS_KEY
from sam3.train.loss.sam3_loss import Sam3LossWrapper
from sam3.train.matcher import BinaryHungarianMatcherV2, BinaryOneToManyMatcher
from sam3.train.data.collator import collate_fn_api

optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

matcher = BinaryHungarianMatcherV2(cost_class=2.0, cost_bbox=5.0, cost_giou=2.0, focal=True)
loss_fns = [
    Boxes(weight_dict={"loss_bbox": 5.0, "loss_giou": 2.0}),
    IABCEMdetr(pos_weight=10.0, weight_dict={"loss_ce": 20.0, "presence_loss": 20.0},
               pos_focal=False, alpha=0.25, gamma=2, use_presence=True, pad_n_queries=200),
    Masks(weight_dict={"loss_mask": 200.0, "loss_dice": 10.0},
          focal_alpha=0.25, focal_gamma=2.0, compute_aux=False),
]
o2m_matcher = BinaryOneToManyMatcher(alpha=0.3, threshold=0.4, topk=4)
loss_wrapper = Sam3LossWrapper(
    loss_fns_find=loss_fns, matcher=matcher, o2m_matcher=o2m_matcher, o2m_weight=2.0,
    use_o2m_matcher_on_o2m_aux=False, normalization="local",
    normalize_by_valid_object_num=False,
)

def collate_fn(batch):
    return collate_fn_api(batch, dict_key="input", with_seg_masks=True)

def move_to_device(obj, device):
    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    if isinstance(obj, list):
        return [move_to_device(x, device) for x in obj]
    if isinstance(obj, tuple):
        return tuple(move_to_device(x, device) for x in obj)
    if isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    if hasattr(obj, "__dataclass_fields__"):
        for f in obj.__dataclass_fields__:
            setattr(obj, f, move_to_device(getattr(obj, f), device))
        return obj
    return obj

## 8. Training loop

One shared loop for GPU and TPU. `tpu_utils.autocast_context` gives bf16 autocast when
`mixed_precision="bf16"`, and `tpu_utils.mark_step()` dispatches the XLA graph each step
(a no-op on GPU/CPU). The matcher-index computation and loss follow
`train_sam3_lora_native.py` exactly.

In [ ]:
from torch.utils.data import DataLoader
from pathlib import Path
import json as _json
from tqdm.auto import tqdm

pin = (device.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          collate_fn=collate_fn, num_workers=cfg.num_workers, pin_memory=pin)
val_loader = (DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         collate_fn=collate_fn, num_workers=cfg.num_workers, pin_memory=pin)
              if val_ds is not None else None)

def add_matcher_indices(outputs_list, find_targets):
    with SAM3Output.iteration_mode(outputs_list, iter_mode=SAM3Output.IterMode.ALL_STEPS_PER_STAGE) as it:
        for stage_outputs, stage_targets in zip(it, find_targets):
            for outputs in stage_outputs:
                outputs["indices"] = matcher(outputs, stage_targets)
                if "aux_outputs" in outputs:
                    for aux in outputs["aux_outputs"]:
                        aux["indices"] = matcher(aux, stage_targets)

def run_batch(input_batch):
    input_batch = move_to_device(input_batch, device)
    with tpu_utils.autocast_context(device, use_autocast):
        outputs_list = model(input_batch)
    find_targets = [unwrapped.back_convert(t) for t in input_batch.find_targets]
    for t in find_targets:
        for k, v in t.items():
            if isinstance(v, torch.Tensor):
                t[k] = v.to(device)
    add_matcher_indices(outputs_list, find_targets)
    with tpu_utils.autocast_context(device, use_autocast):
        loss_dict = loss_wrapper(outputs_list, find_targets)
    return loss_dict[CORE_LOSS_KEY]

out_dir = Path(cfg.output_dir); out_dir.mkdir(parents=True, exist_ok=True)
best_val = float("inf")

for epoch in range(cfg.num_epochs):
    model.train()
    tr_losses = []
    for batch in tqdm(train_loader, desc=f"epoch {epoch+1}/{cfg.num_epochs} [train]"):
        loss = run_batch(batch["input"])
        optimizer.zero_grad()
        loss.backward()
        if cfg.grad_clip:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
        optimizer.step()
        tpu_utils.mark_step()               # dispatch XLA graph (no-op off TPU)
        tr_losses.append(float(loss.item()))
    avg_tr = sum(tr_losses) / max(len(tr_losses), 1)

    avg_val = None
    if val_loader is not None:
        model.eval(); vl = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"epoch {epoch+1} [val]"):
                loss = run_batch(batch["input"])
                tpu_utils.mark_step()
                vl.append(float(loss.item()))
        avg_val = sum(vl) / max(len(vl), 1)

    msg = f"epoch {epoch+1}: train_loss={avg_tr:.4f}"
    if avg_val is not None:
        msg += f"  val_loss={avg_val:.4f}"
    print(msg)

    save_lora_weights(model, str(out_dir / "last_lora_weights.pt"))
    if avg_val is not None and avg_val < best_val:
        best_val = avg_val
        save_lora_weights(model, str(out_dir / "best_lora_weights.pt"))
        print(f"  ✓ new best (val_loss={avg_val:.4f})")
    with open(out_dir / "train_log.jsonl", "a") as f:
        f.write(_json.dumps({"epoch": epoch + 1, "train_loss": avg_tr, "val_loss": avg_val}) + "\n")

# ensure a best checkpoint exists even without validation
if not (out_dir / "best_lora_weights.pt").exists():
    import shutil; shutil.copy(out_dir / "last_lora_weights.pt", out_dir / "best_lora_weights.pt")
print("Training complete. LoRA weights in", out_dir)

## 9. Qualitative inference

Run the fine-tuned model on a held-out image and overlay the predicted nucleus masks. SAM3
outputs one mask per object query; we keep queries above `score_threshold` and upsample
their masks to the original image size.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def predict(model, rec, cfg, device, score_threshold=0.5):
    pil = rec["image"].convert("RGB")
    orig_w, orig_h = pil.size
    pil_r = pil.resize((cfg.resolution, cfg.resolution), PILImage.BILINEAR)
    tfm = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
                      v2.Normalize(mean=[0.5]*3, std=[0.5]*3)])
    img_t = tfm(pil_r)

    meta = InferenceMetadata(coco_image_id=0, original_image_id=0, original_category_id=0,
                             original_size=(orig_h, orig_w), object_id=-1, frame_index=-1)
    query = FindQueryLoaded(query_text=cfg.query_text, image_id=0, object_ids_output=[],
                            is_exhaustive=True, query_processing_order=0, inference_metadata=meta)
    dp = Datapoint(find_queries=[query],
                   images=[Image(data=img_t, objects=[], size=(cfg.resolution, cfg.resolution))],
                   raw_images=[pil_r])
    batch = collate_fn_api([dp], dict_key="input", with_seg_masks=True)
    input_batch = move_to_device(batch["input"], device)

    model.eval()
    with tpu_utils.autocast_context(device, use_autocast):
        outputs = model(input_batch)[-1]
    tpu_utils.mark_step()

    scores = outputs["pred_logits"].sigmoid().squeeze(-1)[0]        # [Q]
    masks = outputs["pred_masks"][0]                               # [Q, h, w]
    keep = scores > score_threshold
    scores, masks = scores[keep], masks[keep]
    if masks.numel() == 0:
        return pil, np.zeros((orig_h, orig_w), bool), 0
    masks = F.interpolate(masks.sigmoid().unsqueeze(1).float(), size=(orig_h, orig_w),
                          mode="bilinear", align_corners=False).squeeze(1)
    union = (masks > 0.5).any(0).cpu().numpy()
    return pil, union, int(keep.sum())

_src = hf_val if hf_val is not None else hf_train
_rec = _src[0]
_img, _pred_union, _n = predict(model, _rec, cfg, device, score_threshold=0.5)
print(f"kept {_n} predicted nuclei above threshold")

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(_img); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(_rec["mask"], cmap="gray"); ax[1].set_title("ground-truth mask"); ax[1].axis("off")
ax[2].imshow(_img)
ax[2].imshow(np.ma.masked_where(~_pred_union, _pred_union), cmap="autumn", alpha=0.6)
ax[2].set_title(f"predicted nuclei ({_n})"); ax[2].axis("off")
plt.tight_layout(); plt.show()

## Next steps

- **Full training**: set `MAX_TRAIN_SAMPLES = None`, raise `NUM_EPOCHS` (100 in the MedSAM3
  configs), and increase `BATCH_SIZE` to what your device fits.
- **TPU**: set `INSTALL_TPU = True` and `cfg.device = "tpu"`. For large models under memory
  pressure, enable `quantize_base_model = True` (QLoRA-style int8/int4 of the frozen base;
  LoRA adapters stay full precision).
- **True instance masks**: swap the connected-components step for the original per-nucleus
  masks from [BBBC038](https://bbbc.broadinstitute.org/BBBC038) if you need to separate
  touching nuclei.
- **Evaluation**: use the repo's `validate_sam3_lora.py` for full mAP / cgF1 metrics with NMS
  (this notebook tracks loss + a qualitative overlay only).